# 90 · Prueba: GLM-4.6V-Flash (Z.ai, gratis) en vez de Gemini

**Pregunta:** ¿GLM-4.6V-Flash propone lo bastante bien como para usarlo de asistente (triaje, rasgos v14 y recuadro) y no gastar la cuota de Gemini (500 consultas/día)?

**Cómo:** se corre GLM con los **mismos prompts** sobre fotos que ya están anotadas a mano y que ya pasó Gemini:
- **A1 (`pilot_a1`):** anotación a ciegas → conteo, lado, erosión y calidad: GLM vs Gemini vs persona, en las mismas fotos. También el triaje y el tiempo por consulta.
- **A2 (`a2_test`):** tiene recuadros dibujados a mano → qué tan bien ubica GLM el hueso (IoU).

Las respuestas de GLM van a **archivos propios** (`outputs/<paso>_zai_glm-4.6v-flash_<version>.jsonl`): no tocan las de Gemini ni las anotaciones. Configs: `configs/test_glm_a1.yaml` y `test_glm_a2.yaml`.

**Antes de empezar (una vez):**
1. Crear una cuenta en **Z.ai** (https://z.ai) y una clave de API (*API Keys*). GLM-4.6V-Flash es gratis; límite informado ~1 consulta por segundo.
2. Guardarla en los secretos de Colab (ícono de la llave, a la izquierda) con el nombre **`ZAI_API_KEY`** y darle acceso al notebook.

No gasta cuota de Gemini.

In [ ]:
# --- Preparacion (correr siempre primero) ---
# Monta Drive, clona o actualiza el repo y carga el codigo recien bajado.
from google.colab import drive, output, userdata
drive.mount('/content/drive')
output.enable_custom_widget_manager()          # interfaces (ipywidgets) en Colab

import os, sys, importlib
if not os.path.exists('/content/sepia-inaturalist'):     # sesion nueva: clonar el repo
    token = userdata.get('GITHUB_TOKEN')
    !git clone -q https://{token}@github.com/hugozeballos/sepia-inaturalist.git /content/sepia-inaturalist
%cd /content/sepia-inaturalist
!git pull -q
!git log -1 --oneline
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')   # nunca en el codigo
os.environ['ZAI_API_KEY'] = userdata.get('ZAI_API_KEY')         # clave de Z.ai (GLM)
os.environ['PYTHONPATH'] = '/content/sepia-inaturalist/src'     # para los !python scripts/...

sys.path.insert(0, '/content/sepia-inaturalist/src')
for m in [m for m in sys.modules if m == 'sepia' or m.startswith('sepia.')]:
    del sys.modules[m]                          # descartar la version vieja en memoria
sys.path_importer_cache.pop('/content/sepia-inaturalist/src', None)
importlib.invalidate_caches()
import sepia

# Los 4 prompts que se usan desde el 4 nov (detalle en docs/prompt_versions.md)
VERSIONS = ['photo_traits_v14', 'photo_traits_v13', 'photo_traits_v12', 'photo_traits_v8']
ANNOTATOR = 'HZ'
BONEBOX = 'bonebox_v1'   # recuadro del hueso (solo fotos con 1 hueso segun v14)

In [ ]:
import pandas as pd
from sepia.annotate import load_events

cfg_gem = sepia.load_config('configs/pilot_a1.yaml')      # Gemini (ya corrido)
cfg_glm = sepia.load_config('configs/test_glm_a1.yaml')   # GLM, mismas fotos
V = 'photo_traits_v14'
N_REC = 15                                                # registros de prueba (~25 fotos)

# Registros de A1 con anotacion manual (a ciegas) y con resultados de Gemini v14
ind = load_events(cfg_gem)['individual']
gem_v14 = sepia.load_photo_traits(cfg_gem, V)
ok = sorted(set(ind['observation_id'].astype(int)) & set(gem_v14['observation_id'].astype(int)))
IDS = ok[:N_REC]
IDS_STR = ','.join(map(str, IDS))
obs, pho = sepia.load_tables(cfg_gem)
PHOTOS = sorted(pho.loc[pho['observation_id'].isin(IDS), 'photo_id'].astype(int))
print(f"{len(ok)} registros anotados con Gemini v14 · prueba: {len(IDS)} registros, {len(PHOTOS)} fotos")
print(IDS_STR)

## 1. ¿Funciona la API? (1 consulta)

Si falla: revisar la clave `ZAI_API_KEY`. Si dice que el modelo no existe, revisar el nombre exacto en la consola de Z.ai y cambiar `model:` en `configs/test_glm_a1.yaml`.

In [ ]:
!python scripts/triage.py --config configs/test_glm_a1.yaml --ids {IDS[0]}

## 2. Correr GLM: triaje, v14 y recuadro

Una consulta por registro (triaje y v14) y una por foto con 1 hueso (recuadro). Reanudable: si se corta, volver a correr la celda.

In [ ]:
!python scripts/triage.py --config configs/test_glm_a1.yaml --ids {IDS_STR}
!python scripts/photo_traits.py --config configs/test_glm_a1.yaml --prompt-version {V} --ids {IDS_STR}
!python scripts/bonebox.py --config configs/test_glm_a1.yaml --ids {IDS_STR}

## 3. Resultados: GLM vs Gemini, contra la anotación manual (mismas fotos)

- **count**: huesos visibles por foto; **side / erosion / photo_quality**: fotos con 1 hueso.
- *side when decided*: acierto del lado cuando el modelo no dice `unclear`.
- Errores: respuestas que no cumplieron el JSON (GLM no lo garantiza como Gemini).

In [ ]:
from sepia.io import read_model_outputs

glm_v14 = sepia.load_photo_traits(cfg_glm, V)
both = sorted(set(glm_v14['photo_id'].astype(int)) & set(gem_v14['photo_id'].astype(int)) & set(PHOTOS))
print(f"Fotos con respuesta de los dos modelos: {len(both)}")

KEYS = ['count_agreement', 'side_agreement', 'side_agreement_when_decided', 'side_unclear_model',
        'erosion_agreement', 'photo_quality_agreement', 'count_n', 'side_n']
rows = {}
for name, c in [('Gemini 3.5 Flash-Lite', cfg_gem), ('GLM-4.6V-Flash', cfg_glm)]:
    m = sepia.evaluate_photo_traits(c, V, photo_ids=both, save=False, quiet=True).attrs['metrics']
    rows[name] = {k: m.get(k) for k in KEYS}
res = pd.DataFrame(rows)
display(res.style.format(lambda v: f"{v:.0%}" if isinstance(v, float) else v))

# Tiempo y errores por consulta (GLM)
for kind, ver in [('triage', 'triage_v2'), ('photo_traits', V), ('bonebox', 'bonebox_v1')]:
    raw = read_model_outputs(cfg_glm, kind, 'zai', 'glm-4.6v-flash', ver)
    if raw.empty:
        print(f"{kind}: sin resultados"); continue
    last = raw.sort_values('saved_at').drop_duplicates('photo_id' if 'photo_id' in raw else 'observation_id', keep='last')
    print(f"{kind}: {len(last)} consultas · con error {int(last['error'].notna().sum())} · "
          f"segundos por consulta (mediana) {last['elapsed_s'].median() if 'elapsed_s' in last else '—'}")

In [ ]:
# Lado: tabla persona (filas) x modelo (columnas), para cada modelo
for name, c in [('Gemini', cfg_gem), ('GLM', cfg_glm)]:
    m = sepia.evaluate_photo_traits(c, V, photo_ids=both, save=False, quiet=True).attrs['metrics']
    print(f"\n{name} · lado"); display(pd.DataFrame(m.get('side_confusion', {})).T.fillna(0).astype(int))

# Triaje: estado y numero de huesos por registro, GLM vs Gemini
tg = sepia.triage.load_triage(cfg_gem, 'gemini', cfg_gem.triage['model'], 'triage_v2')
tz = sepia.triage.load_triage(cfg_glm, 'zai', 'glm-4.6v-flash', 'triage_v2')
t = tg[['observation_id', 'record_state', 'n_cuttlebones']].merge(
    tz[['observation_id', 'record_state', 'n_cuttlebones']], on='observation_id', suffixes=('_gemini', '_glm'))
t = t[t['observation_id'].isin(IDS)]
letters = ind[ind['observation_id'].isin(IDS)].groupby('observation_id')['individual'].nunique().rename('n_manual')
t = t.merge(letters, on='observation_id', how='left')
print(f"\nTriaje en {len(t)} registros · estado igual: {(t.record_state_gemini == t.record_state_glm).mean():.0%} · "
      f"huesos = manual: Gemini {(t.n_cuttlebones_gemini == t.n_manual).mean():.0%}, "
      f"GLM {(t.n_cuttlebones_glm == t.n_manual).mean():.0%}")
display(t)

## 4. Recuadros de GLM en A1 (a la vista)

A1 no tiene recuadros a mano: se miran. **Rojo** = leído como pide el prompt (`[ymin, xmin, ymax, xmax]`); **azul** = orden invertido (`[xmin, ymin, xmax, ymax]`, el habitual de GLM). El que calce con el hueso dice qué orden usa GLM.

In [ ]:
from PIL import Image, ImageDraw, ImageOps
from sepia.bonebox import load_bonebox, box2d_to_crop
from IPython.display import display as show

bb = load_bonebox(cfg_glm)
bb = bb[bb['photo_id'].isin(PHOTOS)]
d1024 = sepia.paths(cfg_glm)['base'] / 'photos' / '1024px'
thumbs = []
for r in bb.itertuples(index=False):
    f = pho.loc[pho['photo_id'] == r.photo_id, 'file_name'].iloc[0]
    im = Image.open(d1024 / (f.rsplit('.', 1)[0] + '.jpg')).convert('RGB')
    W, H = im.size; dr = ImageDraw.Draw(im)
    b = r.box_2d if isinstance(r.box_2d, list) and len(r.box_2d) == 4 else None
    for box, col in [(box2d_to_crop(b) if b else None, 'red'),
                     (box2d_to_crop([b[1], b[0], b[3], b[2]]) if b else None, 'blue')]:
        if box:
            dr.rectangle([box['x0'] * W, box['y0'] * H, box['x1'] * W, box['y1'] * H], outline=col, width=6)
    im.thumbnail((300, 300)); thumbs.append(im)
if thumbs:
    cols = 5; rows_ = -(-len(thumbs) // cols)
    grid = Image.new('RGB', (cols * 305, rows_ * 305), 'white')
    for i, im in enumerate(thumbs):
        grid.paste(im, ((i % cols) * 305, (i // cols) * 305))
    show(grid)
print(f"{len(thumbs)} fotos con respuesta de recuadro")

## 5. Recuadros de GLM contra recuadros a mano (A2)

`a2_test` tiene recuadros dibujados en la anotación. Se corre el recuadro de GLM en hasta 20 de esas fotos y se mide el **IoU** (0 = no se tocan, 1 = idénticos; ≥ 0,5 se suele contar como acierto), en los dos órdenes posibles.

In [ ]:
cfg_a2 = sepia.load_config('configs/test_glm_a2.yaml')
ind2 = load_events(cfg_a2)['individual']
one = ind2.groupby('photo_id').filter(lambda g: len(g) == 1)
with_box = sorted(one.loc[one['crop_box'].apply(lambda b: isinstance(b, dict)), 'photo_id'].astype(int))[:20] \
    if 'crop_box' in one else []
print(f"Fotos de A2 con 1 individuo y recuadro a mano: {len(with_box)}")
if with_box:
    sepia.run_bonebox(cfg_a2, photo_ids=with_box)
    cmp = sepia.bonebox.compare_with_manual(cfg_a2)
    for col in ['iou_yx', 'iou_xy']:
        v = cmp[col].dropna()
        print(f"{col}: IoU mediana {v.median():.2f} · aciertos (IoU >= 0,5) {(v >= 0.5).mean():.0%} ({len(v)} fotos)")
    display(cmp)

## 6. Cómo decidir

- **Recuadro:** si el IoU (en el orden que use GLM) tiene mediana ≥ 0,7 y ≥ 80 % de aciertos, GLM sirve para `bonebox`. Si usa el orden invertido, se hace un prompt `bonebox` con ese orden (versión nueva).
- **Triaje y conteo:** si coincide con la persona casi como Gemini, GLM sirve para el triaje.
- **Lado, erosión, calidad:** si GLM queda muy por debajo de Gemini, conviene dejar v14 en Gemini (500/día) y usar GLM solo para triaje y recuadro.
- **Errores de JSON:** si son muchos, GLM se vuelve molesto aunque acierte.

Son ~25 fotos: diferencias de ±10 % no son significativas. Sirve para descartar o seguir, no para afinar.